In [1]:
from unittest import skip

import pandas as pd
import numpy as np

# Daten einlesen

In [2]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")
df.head()

,anr,datum,legisjahr,jahrzehnt,rechtsform_name,titel_kurz_d,anzahl,beteiligung,annahme,volkja-proz,...,sg-japroz,gr-japroz,ag-japroz,tg-japroz,ti-japroz,vd-japroz,vs-japroz,ne-japroz,ge-japroz,ju-japroz
0,1.0,1848-09-12,1848-1851,1840,Obligatorisches Referendum,Bundesverfassung der schweizerischen Eidgenoss...,1,NaN,1.0,72.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Mass und Gewicht,9,NaN,0.0,50.44,...,26.58,5.62,46.78,84.56,88.62,66.56,48.18,89.41,87.64,NaN
2,3.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Gleichstellung der Juden und Naturalisierten m...,9,NaN,1.0,53.23,...,29.87,10.35,63.32,85.27,84.51,62.70,37.89,85.41,85.95,NaN
3,4.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Stimmrecht der Niedergelassenen in Gemeindeang...,9,NaN,0.0,43.08,...,27.99,9.89,60.67,84.98,63.39,10.65,15.91,71.95,37.97,NaN
4,5.0,1866-01-14,1863-1866,1860,Obligatorisches Referendum,Besteuerung und zivilrechtliche Verhältnisse d...,9,NaN,0.0,39.88,...,24.29,10.45,50.71,81.12,59.32,6.79,13.13,74.70,58.28,NaN


## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in df:
    scores = []
    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']
        if col.startswith('p-') or col.endswith('-pos'):

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
                scores.append((50 - ja_proz) / 100)

            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                scores.append(0.0)

            else:
                scores.append(np.nan)

    # Partei- und Positions-Spalten speichern
    if col.startswith('p-') or col.endswith('-pos'):
        neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

df_with_positions.head()

In [13]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)

### Datenset für Heatmap

In [ ]:
df_pos = df.copy() # Kopie des Datensets
parteien_cols = [c for c in df_pos.columns if str(c).startswith("p-")] # Liste der Partei-Spalten
position_cols = ['br-pos','bv-pos']+parteien_cols # verschiedene Positionen zusammenhängen.

canton_cols = [c for c in df.columns if str(c).endswith('-japroz')] # Liste der Kanton-Spalten

rows = [] #leere Liste für die Zeilen

# Dreistufige Iteration über die Zeilen, die Parteien und die Kantone
for idx, row in df_pos.iterrows(): # 1. Iteration Zeile (Abstimmung)
    for partei in parteien_cols: # 2. Iteration Partei
        zeile = {
            "abstimmung": idx, #Spalte Abstimmung
            "partei": partei, #Spalte Partei
        }
        for kanton in canton_cols: # 3. Iteration Kanton
            ja_proz = row[kanton]
            position = row[partei]
            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                zeile[kanton] = (ja_proz - 50) / 100
            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
               zeile[kanton] = (50 - ja_proz) / 100
            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                zeile[kanton] = 0.0

            else:
                zeile[kanton] = np.nan
            
            rows.append(zeile)

df_heatmap = pd.DataFrame(rows)
erg = df_heatmap.groupby("partei").mean(numeric_only=True) # Gruppieren nach Partei und Berechnen des Mittelwerts.
erg.head()

,abstimmung,zh-japroz,be-japroz,lu-japroz,ur-japroz,sz-japroz,ow-japroz,nw-japroz,gl-japroz,zg-japroz,fr-japroz,so-japroz,bs-japroz,bl-japroz,sh-japroz,ar-japroz,ai-japroz,sg-japroz,gr-japroz,ag-japroz,tg-japroz,ti-japroz,vd-japroz,vs-japroz,ne-japroz,ge-japroz,ju-japroz
partei,,,,,,,,,,,,,,,,,,,,,,,,,,,
p-acs,349.5,0.039455,0.059064,0.091100,0.085536,0.121109,0.132373,0.126300,0.094773,0.078936,0.101209,0.091682,0.008845,0.073491,0.076055,0.090909,0.104882,0.088400,-0.004445,0.107273,0.093218,0.066491,0.077736,0.109018,0.104964,0.058127,0.043150
p-bdp,349.5,0.116646,0.105624,0.124942,0.099520,0.100476,0.118520,0.127376,0.096507,0.137486,0.114364,0.100852,0.101039,0.109134,0.078921,0.096216,0.105809,0.101233,0.124783,0.108230,0.098364,0.084327,0.135880,0.119490,0.110600,0.104014,0.088604
p-bpuk,349.5,0.212600,0.168300,0.180700,0.059300,0.066400,0.062900,0.091200,0.165900,0.214100,0.128700,0.194600,0.281100,0.203400,0.131900,0.159700,0.053600,0.143400,0.114800,0.168800,0.185700,0.053200,0.064500,-0.303500,0.176700,0.077500,0.128000
p-eco,349.5,0.096737,0.092256,0.108237,0.091246,0.098527,0.119948,0.125236,0.101854,0.122034,0.099008,0.089703,0.070203,0.090111,0.079941,0.101568,0.131079,0.105841,0.106842,0.099814,0.108543,0.064491,0.098372,0.105874,0.076041,0.076700,0.035506
p-edk,349.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
p-edu,349.5,0.045165,0.053305,0.070183,0.077680,0.093523,0.094761,0.091266,0.076724,0.068939,0.041052,0.056028,0.008943,0.043213,0.060253,0.075007,0.110876,0.076508,0.061362,0.071776,0.085243,0.039116,0.019325,0.059864,0.007478,0.000336,0.004243
p-endk,349.5,0.042600,0.026400,0.062050,-0.002250,-0.002500,0.020050,0.021550,-0.024250,0.055925,0.003475,0.010450,0.055025,0.012550,-0.006350,-0.002350,0.012375,0.008375,0.035925,0.016625,-0.003900,-0.015375,0.008625,-0.004600,0.007675,0.030875,-0.044725
p-evp,349.5,0.090806,0.083406,0.071515,0.059190,0.024118,0.049778,0.055758,0.065019,0.072818,0.073539,0.066193,0.107961,0.086097,0.063951,0.061575,0.052887,0.068731,0.081641,0.056087,0.061629,0.071429,0.077611,0.042792,0.078204,0.092041,0.063239
p-fdk,349.5,0.033167,0.045611,0.062589,0.060700,0.023167,0.077700,0.074522,0.006178,0.088378,0.061633,0.030689,0.066489,0.048011,0.026733,0.005211,0.033022,0.016111,0.070844,0.023878,0.010600,0.035011,0.057422,0.107633,0.070600,0.069211,0.078111


In [20]:
kanton_cols = sorted(
    c for c in df.columns if str(c).endswith("-japroz")
)
akteure = ["br-pos"] + sorted(
    c for c in df.columns if str(c).startswith("p-")
)
kanton_labels = [c.replace("-japroz", "").upper() for c in kanton_cols]


def alignment_score(position, ja_proz):
    pos = pd.to_numeric(position, errors="coerce")
    ja = pd.to_numeric(ja_proz, errors="coerce")
    if pos not in (1.0, 2.0) or pd.isna(ja):
        return np.nan
    if pos == 1.0:
        return (ja - 50) / 100
    return (50 - ja) / 100


# --- eine Abstimmung wählen (Beispiele) ---
# zeile = df.iloc[0]
# zeile = df.loc[df["anr"] == 642.0].iloc[0]

zeile = df.loc[df["anr"] == 642.0].iloc[0]  # anpassen

rows = {}
for actor in akteure:
    pos = zeile[actor]
    rows[actor] = {
        lab: alignment_score(pos, zeile[kcol])
        for kcol, lab in zip(kanton_cols, kanton_labels)
    }

heatmap_df = pd.DataFrame(rows).T
heatmap_df.index.name = "akteur"

# optional: nur Akteure mit mindestens einem gültigen Wert
heatmap_df = heatmap_df.dropna(how="all")

heatmap_df

,AG,AI,AR,BE,BL,BS,FR,GE,GL,GR,...,SH,SO,SZ,TG,TI,UR,VD,VS,ZG,ZH
akteur,,,,,,,,,,,,,,,,,,,,,
br-pos,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-eco,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-edu,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-evp,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-fdp,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-gem,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-gps,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-kvp,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-lega,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
